# Notebook 3 — Agent Demo

This notebook loads the LangChain agent defined in `agent/agent.py` and runs a set of example queries that demonstrate the full pipeline in action.

Each query shows:
- Which tools the agent decided to invoke
- What data those tools returned
- The final synthesised answer

This is the showpiece notebook — the outputs here are what goes into your report appendix.

> Prerequisite: `02_pipeline.ipynb` must have run. Docker services must be up.

## 0. Setup — load the agent

In [1]:
import sys, os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')

# Add the project root to the Python path so we can import agent/
sys.path.insert(0, os.path.abspath('..'))

from agent.agent import build_agent

# Build the agent — this wires together the LLM, tools, and system prompt
agent_executor = build_agent(verbose=True)

print('Agent loaded and ready')

c:\UCL\financial-signal-agent\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\UCL\financial-signal-agent\venv\Lib\site-packages\langchain\agents\__init__.py)

---
## 1. Helper — run a query and display the reasoning trace

We wrap `agent_executor.invoke()` in a small helper that prints the intermediate steps clearly. This makes the agent's reasoning transparent for the report.

In [ ]:
def run_query(question: str):
    """Run a question through the agent and print the full reasoning trace."""
    print('=' * 60)
    print(f'QUERY: {question}')
    print('=' * 60)

    result = agent_executor.invoke({'input': question})

    # Intermediate steps show which tools were called and what they returned
    print('\nREASONING TRACE:')
    for step in result.get('intermediate_steps', []):
        action, observation = step
        print(f'  Tool called : {action.tool}')
        print(f'  Input       : {action.tool_input}')
        print(f'  Output      : {str(observation)[:300]}...')
        print()

    print('FINAL ANSWER:')
    print(result['output'])
    print()
    return result

---
## 2. Demo Query 1 — Price trend analysis

A purely structured query. The agent should call `query_price_data` only, using SQL to compute the answer from PostgreSQL.

In [ ]:
r1 = run_query(
    'What is the 30-day average closing price and current RSI for NVIDIA (NVDA)? '
    'Is it currently overbought or oversold?'
)

---
## 3. Demo Query 2 — Sentiment vs price divergence

A multi-tool query requiring both structured price data and unstructured news sentiment. The agent should call `query_price_data` AND `search_financial_news` and synthesise the results.

In [ ]:
r2 = run_query(
    'Is there a divergence between NVIDIA news sentiment and its 30-day price momentum? '
    'Summarise what the news is saying and compare it to the price trend.'
)

---
## 4. Demo Query 3 — SEC filing risk analysis

A document-retrieval query. The agent should call `search_financial_news` (which searches SEC filings too) and `get_filing_metadata` to ground its answer in disclosed risk factors.

In [ ]:
r3 = run_query(
    'What key risks did Apple (AAPL) disclose in its most recent 10-K filing? '
    'Cross-reference these risks with AAPL price volatility over the last 3 months.'
)

---
## 5. Demo Query 4 — Sector comparison

A broader analytical query requiring the agent to aggregate across multiple tickers. Tests whether the agent constructs correct SQL GROUP BY logic.

In [ ]:
r4 = run_query(
    'Compare the 90-day price performance of the banking sector (JPM, GS, BAC) '
    'versus the tech sector (AAPL, MSFT, NVDA). Which sector has shown stronger momentum?'
)

---
## 6. Hallucination check — verifying agent citations

A key concern with LLM agents is hallucination — generating plausible-sounding but ungrounded answers. We verify that the agent's factual claims can be traced back to a specific tool call and data source.

In [ ]:
# Extract all tool calls from the four demo queries and verify each has a traceable source
all_results = [r1, r2, r3, r4]
query_labels = ['Price trend', 'Sentiment divergence', 'SEC risk', 'Sector comparison']

print('TOOL USAGE AUDIT')
print('=' * 60)
for label, result in zip(query_labels, all_results):
    steps = result.get('intermediate_steps', [])
    tools_used = [s[0].tool for s in steps]
    print(f'{label}')
    print(f'  Tools invoked : {tools_used}')
    print(f'  Steps         : {len(steps)}')
    # A response with zero tool calls is a red flag — the agent answered from training data,
    # not from the pipeline, which counts as hallucination in this context
    if not tools_used:
        print('  WARNING: no tools called — answer may not be grounded in pipeline data')
    print()

---
## 7. Interactive query cell

Use this cell to test your own questions against the agent.

In [ ]:
# Change this question and re-run the cell to test any query
your_question = 'Which stock in our watchlist has the highest RSI right now?'

run_query(your_question)